[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/bonelab_inverse_dynamics/blob/main/notebooks/p1_report.ipynb)

# Subject P1 Inverse Dynamics — step by step

The notebook form of `reports/p1_report.html`
(`uv run python -m boneid.report v3d`): the
whole-body pipeline (twelve segments, eleven joints) on real treadmill
data, validated against Visual3D.

## How to run this notebook

```bash
uv run jupyter lab notebooks/p1_report.ipynb          # interactive
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/p1_report.ipynb
```

The second form is what CI does: it runs every cell top to bottom and writes the
outputs back into the file, so a clean execution is the test.

## How to convert it to a script

```bash
uv run jupyter nbconvert --to script notebooks/p1_report.ipynb
```

which writes `notebooks/p1_report.py` — the same cells, minus the prose, runnable
with `uv run python notebooks/p1_report.py`.

## What is *not* in here

No mathematics. Every number below comes from `boneid.core`, `boneid.io_v3d`
and `boneid.report`; the notebook only calls them, arranges the figures and
narrates. It needs the local validation file
`p1_5StridesData.mat` (path in the next cell).


In [1]:
# Colab bootstrap — a no-op when running locally in the uv environment.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "boneid @ git+https://github.com/jeremydwong/bonelab_inverse_dynamics"],
                   check=True)
os.makedirs("../reports", exist_ok=True)  # where viewer/report HTML lands

# Data: on Colab, paste a DIRECT download link to p1_5StridesData.mat below
# (e.g. a Dropbox share link with ?dl=1). Locally the Dropbox path is used.
DATA_URL = ""
if IN_COLAB:
    assert DATA_URL, "Set DATA_URL to a direct link to p1_5StridesData.mat"
    if not os.path.exists("p1_5StridesData.mat"):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL, "p1_5StridesData.mat")
        print("downloaded p1_5StridesData.mat")


## Setup

The dataset is van der Zee, Mundinger & Kuo, *A biomechanics dataset of
healthy human walking at various speeds, step lengths and step widths*,
Scientific Data (2022),
<https://www.nature.com/articles/s41597-022-01817-1> — 33 controlled
**walking** conditions (speed × step length × step width). There is no
running anywhere in it.

**Filtering, stated once.** Force and COP are low-passed at 50 Hz inside
`io_v3d.ground_wrench` as an anti-alias step before being interpolated
to the 120 Hz mocap clock, so `force_lowpass_hz=0` here — otherwise the
wrench would be filtered twice. Markers are the raw `_pos` targets and
the only kinematic filter is the 12 Hz low-pass on segment poses inside
`core.chain_kinematics`.

In [2]:
import numpy as np

from boneid import io_v3d as io
from boneid.core import (chain_kinematics, detect_contact,
                         energy_audit_whole_body,
                         inverse_dynamics_whole_body)

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from boneid import report as R
from boneid.report import (JOINT_COLORS, FAINT, NEUTRAL, new_fig, new_grid,
                           style_axes)


def show(fig):
    """Render a matplotlib figure inline as an SVG, via the report's own
    `fig_svg` — identical output to the HTML report, no second code path.

    (`boneid.report` forces the Agg backend at import so report generation
    works headless; going through `fig_svg` sidesteps the backend entirely.)"""
    display(HTML(f'<img src="{R.fig_svg(fig)}" style="max-width:100%">'))

In [3]:
import os
PATH = R.V3D_PATH if os.path.exists(R.V3D_PATH) else "p1_5StridesData.mat"
TRIAL = R.V3D_TRIAL        # slot 13

params = R.p1_params()
trials = io.load_v3d_trials(PATH)
trial = next(tr for tr in trials if tr.index == TRIAL)
print(f"{len(trials)} populated trials of 33 slots")
print(f"analysing slot {trial.index}: {len(trial.t) / trial.rate:.1f} s "
      f"at {trial.rate:.0f} Hz mocap / {trial.analog_rate:.0f} Hz analog")
print(params)

28 populated trials of 33 slots
analysing slot 13: 5.3 s at 120 Hz mocap / 1200 Hz analog
AnalysisParams(gravity=array([ 0.  ,  0.  , -9.81]), contact_threshold_n=20.0, lowpass_hz=12.0, force_lowpass_hz=0.0, filter_order=4, treadmill_speed=0.0, flag_crossover=False)


## Step 1 — the speed sweep, read from the data

The condition labels are not in the export, so belt speed is measured
from the stance-phase heel marker and the stride/stance times from the
force threshold. `trial_sweep_stats` does all 28 trials.

In [4]:
sweep = R.trial_sweep_stats(trials, params.contact_threshold_n)
body_mass_all = sweep["body_mass"]
print(f"belt speed   {sweep['speed'].min():.2f} – {sweep['speed'].max():.2f} m/s")
print(f"stride time  {sweep['stride_time'].min():.2f} – "
      f"{sweep['stride_time'].max():.2f} s")
print(f"duty factor  {sweep['duty'].min():.2f} – {sweep['duty'].max():.2f} "
      f"(no aerial phase anywhere -> walking, not running)")
print(f"body mass    {body_mass_all.min():.1f} – {body_mass_all.max():.1f} kg")

bw = body_mass_all.mean() * 9.81
fig, axes = new_fig(2, height=3.0)
order = np.argsort(sweep["index"])
x = np.arange(len(order))
axes[0].plot(x, sweep["speed"][order], "o-", color=JOINT_COLORS[0], lw=1.2,
             ms=4, label="belt speed (m/s)")
axes[0].plot(x, sweep["stride_time"][order], "s-", color=JOINT_COLORS[1],
             lw=1.2, ms=4, label="stride time (s)")
axes[0].set_xticks(x[::4])
axes[0].set_xticklabels([f"{int(i)}" for i in sweep["index"][order][::4]])
style_axes(axes[0], "trial slot in file", "")
axes[0].legend(frameon=False, fontsize=8.5)
axes[1].plot(sweep["speed"], sweep["peak_fz"] / bw, "o", ms=5,
             color=JOINT_COLORS[2], alpha=0.8)
style_axes(axes[1], "belt speed (m/s)", "peak vertical GRF (BW)")
show(fig)

belt speed   0.65 – 1.87 m/s
stride time  0.72 – 1.84 s
duty factor  0.55 – 0.65 (no aerial phase anywhere -> walking, not running)
body mass    84.6 – 85.1 kg


## Step 2 — contact detection, both feet

Which belt carries which foot is re-derived per side and per trial from
COP-to-marker proximity (`io_v3d.plate_for_side`), never assumed.

In [5]:
plate = {s: io.plate_for_side(trial, s, params.contact_threshold_n)
         for s in R.SIDES}
fz = io.analog_to_mocap(trial, trial.force[:, :, 2])
mask = {s: detect_contact(fz[:, plate[s]], params.contact_threshold_n,
                          min_gap=12)[0] for s in R.SIDES}
xover = io.crossover_flags(trial, "r", params.contact_threshold_n)
t = trial.t
print(f"right foot on plate {plate['r'] + 1}, left on plate {plate['l'] + 1}")
print(f"double support: {100 * xover.mean():.0f}% of frames")

span = slice(0, int(4.0 * trial.rate))
fig, ax = new_fig(height=3.0)
for s, c in (("r", 0), ("l", 2)):
    for k, (a, b) in enumerate(R.contact_spans(mask[s][span], t[span])):
        ax.axvspan(a, b, color=JOINT_COLORS[c], alpha=0.10, lw=0,
                   label=f"{s} contact" if k == 0 else None)
    ax.plot(t[span], fz[span, plate[s]], color=JOINT_COLORS[c], lw=1.6,
            label=f"plate {plate[s] + 1} ({s})")
ax.axhline(params.contact_threshold_n, color=NEUTRAL, lw=1.0, ls="--")
style_axes(ax, "time (s)", "vertical force (N)")
ax.legend(frameon=False, fontsize=8.5, ncol=2)
show(fig)

right foot on plate 2, left on plate 1
double support: 21% of frames


## Step 3 — the skeleton, measured

`two_leg_chain` calls `io_v3d.build_chain` for each side: body mass from
the mean total vertical GRF over whole strides, segment lengths from the
trial-mean joint-centre distances, de Leva (1996) regressions for masses,
COM offsets and inertias. Nothing about the subject is hardcoded.

The two chains share one pelvis segment — it must not be counted twice
when tallying modelled mass, which is what `unmodelled_weight` handles.

In [6]:
chain, skel_u, kin_u = R.whole_body_chain(trial, params)
skel_r, kin_r, ground_r, skel_l, kin_l, ground_l = chain
body_mass, modelled_mass, unmod_n = R.unmodelled_weight(trial, skel_r, skel_l,
                                                        params)
bw = body_mass * 9.81
print(f"body mass          {body_mass:.2f} kg   ({bw:.0f} N)")
upper_mass = float(skel_u.mass.sum())
print(f"legs + pelvis      {modelled_mass:.2f} kg "
      f"({100 * modelled_mass / body_mass:.0f}%)")
print(f"upper body         {upper_mass:.2f} kg "
      f"({100 * upper_mass / body_mass:.0f}%)  <- step 6 checks the L5S1 "
      f"wrench against its weight, {upper_mass * 9.81:.0f} N")
display(HTML(R.deleva_table_html(skel_r, body_mass)))
display(HTML(R.deleva_table_html(skel_u, body_mass)))

body mass          84.96 kg   (833 N)
legs + pelvis      43.23 kg (51%)
upper body         41.72 kg (49%)  <- step 6 checks the L5S1 wrench against its weight, 409 N


segment,mass frac,mass (kg),length (cm),COM frac,|COM| from origin (cm),rg sag / tra / long,Ixx / Iyy / Izz (kg m²)
foot,0.0137,1.16,22.9,0.4415,5.5,0.257 / 0.245 / 0.124,0.0040 / 0.0037 / 0.0009
shank,0.0433,3.68,39.4,0.4459,17.6,0.255 / 0.249 / 0.103,0.0372 / 0.0355 / 0.0061
thigh,0.1416,12.03,41.1,0.4095,16.8,0.329 / 0.329 / 0.149,0.2196 / 0.2196 / 0.0451
pelvis,0.1117,9.49,11.5,0.6115,7.0,0.615 / 0.551 / 0.587,0.0472 / 0.0379 / 0.0430


segment,mass frac,mass (kg),length (cm),COM frac,|COM| from origin (cm),rg sag / tra / long,Ixx / Iyy / Izz (kg m²)
r_forearm_hand,0.0223,1.89,29.5,0.6690,19.7,0.428 / 0.420 / 0.120,0.0302 / 0.0290 / 0.0024
r_upper_arm,0.0271,2.30,34.9,0.5772,20.1,0.285 / 0.269 / 0.158,0.0228 / 0.0203 / 0.0070
l_forearm_hand,0.0223,1.89,28.2,0.6719,18.9,0.433 / 0.424 / 0.121,0.0282 / 0.0271 / 0.0022
l_upper_arm,0.0271,2.30,34.4,0.5772,19.8,0.285 / 0.269 / 0.158,0.0221 / 0.0197 / 0.0068
torso,0.3923,33.33,44.7,0.6594,29.5,0.462 / 0.413 / 0.271,1.4199 / 1.1375 / 0.4874


## Step 4 — kinematics

Segment frames come straight from anatomical markers; the medio-lateral
axis is flipped for the left leg so both sides are anatomically
consistent. Angular velocity is extracted from dR/dt·Rᵀ, not from
Cardan-angle derivatives (which are only valid for small planar
rotations — the legacy MATLAB's mistake).

In [7]:
ck = chain_kinematics(skel_r, kin_r, params.lowpass_hz, params.filter_order)
joints = skel_r.joint_names

fig, ax = new_fig(height=3.0)
for j in range(3):
    ax.plot(t, kin_r.prox_pos[:, j, 2], color=JOINT_COLORS[j], lw=1.5,
            label=f"R {joints[j]}")
    ax.plot(t, kin_l.prox_pos[:, j, 2], color=JOINT_COLORS[j], lw=1.1,
            ls="--", alpha=0.85, label=f"L {joints[j]}")
style_axes(ax, "time (s)", "joint-centre height (m)")
ax.set_ylim(top=ax.get_ylim()[1] + 0.22 * float(np.ptp(ax.get_ylim())))
ax.legend(frameon=False, fontsize=8.0, ncol=4, loc="upper left")
show(fig)

sl = slice(R.EDGE, -R.EDGE)
w_ref = io.reference_series(trial, "rSkAngVel")
w_ours = ck["omega"][:, 1]
print(f"shank omega (lab x) vs Visual3D: r = "
      f"{np.corrcoef(w_ours[sl, 0], w_ref[sl, 0])[0, 1]:.4f}, "
      f"{np.sqrt(((w_ours[sl, 0] - w_ref[sl, 0]) ** 2).mean()):.2f} rad/s RMS "
      f"on a {np.abs(w_ref[sl, 0]).max():.1f} rad/s peak")

shank omega (lab x) vs Visual3D: r = 0.9985, 0.21 rad/s RMS on a 8.2 rad/s peak


## Step 5 — whole-body inverse dynamics

`inverse_dynamics_two_legs` runs each leg's foot→thigh recursion driven
by its own belt's wrench; each leg's top reaction *is* its hip wrench.
The pelvis then takes **both** hip reactions as distal loads and closes
its own balance; what it leaves is the **L5/S1 joint wrench** (the wrench
the torso exerts on the pelvis). Each arm is its own two-segment
recursion with nothing in the hand, and the torso is balanced against
L5/S1 and both shoulders — leaving the residual at the torso's COM.

Visual3D's own `Kinetic_Kinematic.*ProxEndTorque` is the reference for
all six joints. Naming trap: the shanks are `rSk`/`lSk` for kinematics
but `rSh`/`lSh` for kinetics — `io_v3d.reference_series` hides it.

In [8]:
whole = inverse_dynamics_whole_body(*chain, skel_u, kin_u, params)
legs = {"r": whole.right, "l": whole.left}
refs = {s: R.reference_torques(trial, s) for s in R.SIDES}
rms = R.two_leg_rms(trial, whole)
peak = np.concatenate([R.reference_peaks(trial, s) for s in R.SIDES])

fig, axes = new_grid(2, 3, height=5.0, width=9.0)
for row, side in enumerate(R.SIDES):
    for j in range(3):
        ax = axes[row, j]
        ax.plot(t, refs[side][j][:, 0], color=NEUTRAL, lw=2.8, alpha=0.4,
                label="Visual3D")
        ax.plot(t, legs[side].joint_torque[:, j, 0], color=JOINT_COLORS[j],
                lw=1.2, label="ours")
        style_axes(ax, "time (s)" if row == 1 else "",
                   "torque, lab x (N m)" if j == 0 else "")
        ax.set_title(R.JOINT_LABELS[row * 3 + j], color=NEUTRAL, fontsize=10)
        if row == 0 and j == 0:
            ax.legend(frameon=False, fontsize=8.5)
show(fig)

for k, name in enumerate(R.JOINT_LABELS):
    print(f"{name:8s} RMS {rms[k]:6.2f} N m   peak(V3D) {peak[k]:6.1f} N m"
          f"   {100 * rms[k] / peak[k]:5.1f}%")

R ankle  RMS   3.11 N m   peak(V3D)  175.3 N m     1.8%
R knee   RMS   4.09 N m   peak(V3D)  157.9 N m     2.6%
R hip    RMS  17.32 N m   peak(V3D)  147.5 N m    11.7%
L ankle  RMS   2.53 N m   peak(V3D)  162.6 N m     1.6%
L knee   RMS   4.06 N m   peak(V3D)  159.4 N m     2.5%
L hip    RMS  15.91 N m   peak(V3D)  134.6 N m    11.8%


## Step 6 — the residual, and the L5/S1 joint wrench

Two quantities, where the legs-only model had one. The **L5/S1 joint
wrench** must average minus the weight of everything above it, because
over a whole number of strides that mass has zero mean acceleration —
a number the recursion is never told. The **residual**, now at the
torso COM with every segment modelled, has nothing structural left to
carry and should average near zero.

In [9]:
res_mean = whole.residual_force[sl].mean(axis=0)
l5_mean = whole.l5s1_force[sl].mean(axis=0)
upper_n = float(skel_u.mass.sum()) * 9.81
print(f"mean L5S1 joint F     = ({l5_mean[0]:7.2f}, {l5_mean[1]:7.2f}, "
      f"{l5_mean[2]:8.2f}) N")
print(f"-(upper-body weight)  =                    {-upper_n:8.2f} N")
print(f"difference            =                    "
      f"{l5_mean[2] + upper_n:8.2f} N  "
      f"({100 * abs(l5_mean[2] + upper_n) / upper_n:.2f}% of it)")
print(f"mean torso residual F = ({res_mean[0]:7.2f}, {res_mean[1]:7.2f}, "
      f"{res_mean[2]:8.2f}) N   "
      f"({100 * abs(res_mean[2]) / bw:.3f}% of body weight)")

fig, axes = new_fig(2, height=2.9)
for c, lab in enumerate("xyz"):
    axes[0].plot(t, whole.l5s1_force[:, c], color=JOINT_COLORS[c], lw=1.2,
                 label=f"F{lab}")
    axes[1].plot(t, whole.residual_force[:, c], color=JOINT_COLORS[c], lw=1.2,
                 label=f"F{lab}")
axes[0].axhline(-upper_n, color=NEUTRAL, lw=1.4, ls="--")
axes[0].annotate(f"-(upper-body weight) = {-upper_n:.0f} N", (t[-1], -upper_n),
                 ha="right", va="bottom", fontsize=8.5, color=NEUTRAL)
style_axes(axes[0], "time (s)", "L5S1 joint force (N)")
axes[0].legend(frameon=False, fontsize=8.5, ncol=3)
style_axes(axes[1], "time (s)", "torso residual force (N)")
axes[1].legend(frameon=False, fontsize=8.5, ncol=3)
# RULE: the same quantity in two panels shares one y-scale.
R.share_ylim(axes[0], axes[1])
show(fig)


mean L5S1 joint F     = (   2.36,    2.49,  -405.79) N
-(upper-body weight)  =                     -409.30 N
difference            =                        3.51 N  (0.86% of it)
mean torso residual F = (   0.01,    2.86,    -1.05) N   (0.127% of body weight)


## Step 7 — the energy audit over all twelve segments

d(KE+PE)/dt against eleven joint powers + **both** ground wrench powers
+ the torso residual power. Energy never enters the recursion, so a small
imbalance is a genuine consistency check of the branched model.

In [10]:
audit = energy_audit_whole_body(*chain, skel_u, kin_u, whole, params)
imb = float(np.abs(audit.imbalance[sl]).max())
scale = float(np.abs(audit.de_dt[sl]).max())
print(f"peak imbalance {imb:.1f} W  =  {100 * imb / scale:.2f}% of the "
      f"{scale:.0f} W peak in d(KE+PE)/dt")

fig, axes = new_fig(2, height=2.9)
axes[0].plot(t[sl], audit.de_dt[sl], color=NEUTRAL, lw=2.8, alpha=0.45,
             label="d(KE+PE)/dt")
axes[0].plot(t[sl], audit.power_total[sl], color=JOINT_COLORS[0], lw=1.1,
             label="11 joints + 2 ground + residual")
style_axes(axes[0], "time (s)", "power (W)")
axes[0].legend(frameon=False, fontsize=8.5)
axes[1].plot(t[sl], audit.imbalance[sl], color=JOINT_COLORS[1], lw=1.2)
style_axes(axes[1], "time (s)", "imbalance (W)")
show(fig)

fig, ax = new_fig(height=3.0)
for k in range(6):
    ax.plot(t, audit.joint_power[:, k], color=JOINT_COLORS[k % 3],
            lw=1.5 if k < 3 else 1.1, ls="-" if k < 3 else "--", alpha=0.9,
            label=R.JOINT_LABELS[k])
style_axes(ax, "time (s)", "joint power (W)")
ax.legend(frameon=False, fontsize=8.0, ncol=3)
show(fig)

peak imbalance 6.7 W  =  0.85% of the 781 W peak in d(KE+PE)/dt


## Step 8 — every trial, both legs

`cross_trial_rms` reruns the whole two-leg pipeline on every populated
trial and scores all six joints against Visual3D. It also returns the
mean pelvis residual and the unmodelled weight per trial, so step 6's
check can be repeated across the file.

**Runtime.** The full 28-trial sweep takes only a second or two on this
data, so no subsampling is needed here — but `TRIAL_STRIDE` below is the
knob if you point this notebook at something slower: set it to 4 to run
every fourth trial.

In [11]:
TRIAL_STRIDE = 1          # 1 = all 28 trials; raise it to subsample

subset = trials[::TRIAL_STRIDE]
cross = R.cross_trial_rms(subset, params)
med = np.median(cross["rms"], axis=0)
print(f"{len(cross['index'])} trials x 6 joints = "
      f"{6 * len(cross['index'])} comparisons")
for k, name in enumerate(R.JOINT_LABELS):
    print(f"{name:8s} median RMS {med[k]:6.2f} N m  "
          f"({100 * med[k] / np.median(cross['peak'][:, k]):4.1f}% of median peak)")
gap = cross["residual_fz"] + cross["unmodelled_n"]
print(f"\nresidual check across trials: max |mean Fz + unmodelled weight| = "
      f"{np.abs(gap).max():.1f} N")

fig, axes = new_fig(2, height=3.4)
order = np.argsort(cross["index"])
xs = np.arange(len(order))
for k in range(6):
    right = k < 3
    style = dict(ms=5, color=JOINT_COLORS[k % 3], alpha=0.85, ls="none",
                 mfc=JOINT_COLORS[k % 3] if right else "none", mew=1.3,
                 marker="o" if right else "s")
    axes[0].plot(xs, cross["rms"][order, k], label=R.JOINT_LABELS[k], **style)
    axes[1].plot(cross["speed"], cross["rms"][:, k], **style)
axes[0].set_xticks(xs[::4])
axes[0].set_xticklabels([f"{int(i)}" for i in cross["index"][order][::4]])
style_axes(axes[0], "trial slot in file", "torque RMS vs Visual3D (N m)")
axes[0].legend(frameon=False, fontsize=8.0, ncol=3, loc="upper left")
style_axes(axes[1], "belt speed (m/s)", "")
show(fig)

28 trials x 6 joints = 168 comparisons
R ankle  median RMS   3.16 N m  ( 2.5% of median peak)
R knee   median RMS   3.82 N m  ( 3.7% of median peak)
R hip    median RMS  12.73 N m  (13.4% of median peak)
L ankle  median RMS   2.64 N m  ( 2.1% of median peak)
L knee   median RMS   4.02 N m  ( 4.4% of median peak)
L hip    median RMS  12.22 N m  (12.1% of median peak)

residual check across trials: max |mean Fz + unmodelled weight| = 3.7 N


## Step 9 — the trial, moving

All twelve segments — both legs, the pelvis band, the mass-sized torso
and both arms — with both ground-reaction arrows.
`viz` keys its scene paths on segment names, which are identical on the
two sides, so `p1_viewer_html` draws each leg under its own meshcat root
(`boneid/r`, `boneid/l`) and merges the two animations into one clip
set. Written to a file rather than embedded, to keep this notebook
small.

In [12]:
from pathlib import Path

viewer_path = Path("../reports/p1_viewer.html").resolve()
try:
    html = R.p1_viewer_html(trial, chain, skel_u, kin_u, params)
    viewer_path.write_text(html)
    print(f"wrote {viewer_path} ({viewer_path.stat().st_size / 1e6:.1f} MB)")
    display(HTML(f'<iframe src="{viewer_path.as_uri()}" width="100%" '
                 f'height="460" style="border:1px solid #ddd"></iframe>'))
except Exception as exc:
    print(f"viz skipped: {exc}")

wrote /Users/jeremy/Git/brent/bonelab_inverse_dynamics/reports/p1_viewer.html (2.2 MB)


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/.venv/lib/python3.12/site-packages/IPython/core/display.py:452: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Reproducing the HTML report

`uv run python -m boneid.report v3d` runs exactly the calls above and
writes `reports/p1_report.html`.

In [13]:
powers = R.power_decomposition(trial, chain, skel_u, kin_u, params)
hjc = R.hjc_comparison(trial, params)
html = R.p1_report_html(trial, chain, skel_u, kin_u, whole, audit, params,
                        sweep, cross, powers, hjc)
print(f"{len(html) / 1e6:.1f} MB of report HTML "
      f"(not written here — run the module for that)")


2.4 MB of report HTML (not written here — run the module for that)
